# NFXP Dynamic Demand — Aguirregabiria (2022)

**Monte Carlo simulation and NFXP estimation of the dynamic discrete-choice demand model with two brands, T=52 weeks and homogeneous consumers.**

> Aguirregabiria, V. (2022). *Dynamic demand for differentiated products with fixed-effects unobserved heterogeneity*. Econometrics Journal.

---

### Model overview

Each period $t$ a consumer chooses an action $y_{it} \in \{0, 1, 2\}$:
- $y=0$: **no purchase** (consumer keeps consuming the last brand purchased, which depreciates over time)
- $y=j>0$: **buys brand $j$**

**Endogenous state** $x_{it} = (\ell_{it},\, d_{it})$:
- $\ell_{it}$: last brand purchased
- $d_{it}$: number of periods since last purchase (capped at $D^*$)

**Prices (Hi-Lo, Assumption 2.1):**
$$p_t(j) = z(j) - \text{disc}(j)\cdot e_t(j)$$
Transitory promotion $e_t(j)\in\{0,1\}$ follows an independent two-state Markov chain.

**Deterministic per-period utility function (equation 2.6):**
$$u(y,\ell,d,p;\,\theta) = \begin{cases} \alpha(\ell) + \gamma\,h(\mu) - \beta^{\rm dep}(\ell)\,d & y=0 \\ \alpha(j) + \gamma\,h(\mu - p(j)) - \beta^{\rm sc}(\ell,j) & y=j>0 \end{cases}$$

Idiosyncratic shocks $\varepsilon_{it}(j)$ are i.i.d. Type-I Extreme Value $\Rightarrow$ multinomial logit.

### NFXP estimator
- **Outer loop**: Nelder-Mead optimizer searches over $\theta$
- **Inner loop**: Value Function Iteration (VFI) solves the Bellman equation at each candidate $\theta$; the resulting CCPs are used to evaluate the log-likelihood

Because there is no unobserved heterogeneity here, the full MLE identifies all parameters, including the brand intercepts $\alpha(j)$.

In [39]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Set global seed for reproducibility
SEED = 2024
rng_global = np.random.default_rng(SEED)

## 1. Primitives and true parameters

All model constants are defined here — one place to change when experimenting with the calibration.

In [40]:
# ── Model dimensions ──────────────────────────────────────────────────────────
J     = 2      # number of brands
T     = 52     # periods per consumer (one calendar year of weekly data)
N     = 2_000  # consumers per simulated panel (kept small for NFXP tractability)
D_MAX = 20      # duration cap: Assumption 3.2 in Aguirregabiria (2022)
DELTA = 0.95   # discount factor (calibrated, not estimated)
MU    = 15.0   # consumer disposable income (weekly)

# ── True structural parameters (DGP) ─────────────────────────────────────────
# alpha(1) = 0 is the normalisation; we only set alpha(2) here.
ALPHA_TRUE    = np.array([0.0,  0.30])   # brand intercepts
GAMMA_TRUE    = -0.50                     # price sensitivity (marginal utility of income)
BETA_SC_TRUE  = np.array([[0.00, 0.55],  # beta_sc[k-1,j-1]: switching cost k→j
                           [0.50, 0.00]])
BETA_DEP_TRUE = np.array([0.25, 0.25])   # depreciation rate per brand

# ── Hi-Lo price process (Assumption 2.1) ─────────────────────────────────────
BASE_PRICES   = np.array([10.0, 15.0])   # persistent price component z(j)
PROMO_DISC    = np.array([ 2.0,  2.0])   # discount when brand is on promotion
PROMO_ENTRY   = 0.05   # P(promotion starts | not currently promoted)
PROMO_PERSIST = 0.62   # P(promotion continues | currently promoted)

# ── Monte Carlo settings ──────────────────────────────────────────────────────
MC_REPS = 20    # number of independent replications
MC_SEED = 2024  # master seed

# Parameter name list (matches the theta vector defined below)
PARAM_NAMES = ["alpha_2", "gamma", "beta_sc_12", "beta_sc_21",
               "beta_dep_1", "beta_dep_2"]

# True theta vector in estimator format
THETA_TRUE = np.array([
    ALPHA_TRUE[1],          # alpha_2
    GAMMA_TRUE,             # gamma
    BETA_SC_TRUE[0, 1],     # beta_sc(1→2)
    BETA_SC_TRUE[1, 0],     # beta_sc(2→1)
    BETA_DEP_TRUE[0],       # beta_dep brand 1
    BETA_DEP_TRUE[1],       # beta_dep brand 2
])

print("True parameter vector:")
for name, val in zip(PARAM_NAMES, THETA_TRUE):
    print(f"  {name:<14} = {val}")

True parameter vector:
  alpha_2        = 0.3
  gamma          = -0.5
  beta_sc_12     = 0.55
  beta_sc_21     = 0.5
  beta_dep_1     = 0.25
  beta_dep_2     = 0.25


## 2. Price process — Hi-Lo (Assumption 2.1)

The promotion state $e_t \in \{0,1\}^J$ is a binary vector indicating which brands are on sale. With $J=2$ there are $2^2=4$ possible states. Each brand follows an independent two-state Markov chain, so the joint transition matrix is the outer product of the marginal probabilities.

In [41]:
# All 2^J binary promotion vectors: promo_states[s] = (e_1, e_2, ...)
promo_states = np.array(
    [[(s >> j) & 1 for j in range(J)] for s in range(2 ** J)], dtype=int
)
N_PROMO = len(promo_states)   # = 4 for J=2


def make_promo_transition() -> np.ndarray:
    """
    Build the (N_PROMO × N_PROMO) Markov transition matrix for promotion states.
    Brands' promotions are independent two-state Markov chains, so the joint
    transition probability is the product of the marginal probabilities.
    """
    trans = np.empty((N_PROMO, N_PROMO))
    for s, curr in enumerate(promo_states):
        # Probability each brand is on promotion next period
        prob_on = np.where(curr == 1, PROMO_PERSIST, PROMO_ENTRY)
        for sp, nxt in enumerate(promo_states):
            # Joint probability = product over brands (independence)
            trans[s, sp] = np.prod(np.where(nxt == 1, prob_on, 1.0 - prob_on))
    return trans


PROMO_TRANS = make_promo_transition()   # shape (N_PROMO, N_PROMO)

print("Promotion states (e_1, e_2):")
for s, e in enumerate(promo_states):
    row = PROMO_TRANS[s]
    print(f"  State {s}: e={tuple(e)}  → next-period probs: {np.round(row, 3)}")

print(f"\nRealized prices per state:")
for s, e in enumerate(promo_states):
    prices = BASE_PRICES - PROMO_DISC * e
    print(f"  State {s}: e={tuple(e)}  → p={prices}")

Promotion states (e_1, e_2):
  State 0: e=(0, 0)  → next-period probs: [0.902 0.048 0.048 0.003]
  State 1: e=(1, 0)  → next-period probs: [0.361 0.589 0.019 0.031]
  State 2: e=(0, 1)  → next-period probs: [0.361 0.019 0.589 0.031]
  State 3: e=(1, 1)  → next-period probs: [0.144 0.236 0.236 0.384]

Realized prices per state:
  State 0: e=(0, 0)  → p=[10. 15.]
  State 1: e=(1, 0)  → p=[ 8. 15.]
  State 2: e=(0, 1)  → p=[10. 13.]
  State 3: e=(1, 1)  → p=[ 8. 13.]


## 3. Utility function (equation 2.6)

The deterministic part of the per-period utility function. The stochastic EV shocks $\varepsilon_{it}(j)$ are integrated out analytically in the Bellman equation via the log-sum-exp formula.

In [42]:
def h(c):
    """Log utility of net resources h(c) = log(c); small floor prevents log(0)."""
    return np.log(np.maximum(c, 1e-8))


def flow_util(choice, last_brand, duration, e_idx,
              alpha, gamma, beta_sc, beta_dep) -> float:
    """
    Deterministic per-period utility for one observation (equation 2.6).

    Parameters
    ----------
    choice     : int  — 0 = no purchase, j∈{1,...,J} = buy brand j
    last_brand : int  — ell ∈ {1,...,J}
    duration   : int  — d ∈ {0,...,D_MAX}
    e_idx      : int  — index into promo_states for current promotion vector
    alpha      : (J,) brand intercepts
    gamma      : float price sensitivity
    beta_sc    : (J,J) switching cost matrix, beta_sc[k-1,j-1]
    beta_dep   : (J,) depreciation rates per brand
    """
    l = last_brand - 1                              # convert to 0-indexed
    prices = BASE_PRICES - PROMO_DISC * promo_states[e_idx]

    if choice == 0:
        # No purchase: consumer keeps last brand, which depreciates over time
        return alpha[l] + gamma * h(MU) - beta_dep[l] * duration

    j = choice - 1                                  # convert to 0-indexed
    # Purchase brand j: consumption utility minus any switching cost
    return alpha[j] + gamma * h(MU - prices[j]) - beta_sc[l, j]

## 4. Inner loop: Value Function Iteration (VFI)

VFI is the **core of NFXP**. Given a candidate parameter vector $\theta$ we iterate the Bellman operator:

$$T(V)[\ell,d,e] = \log \sum_{j=0}^{J} \exp\!\bigl(u(j,\ell,d,e;\theta) + \delta \cdot EV(j,\ell,d,e)\bigr)$$

The log-sum-exp is the closed form for $\mathbb{E}[\max_j(Q_j + \varepsilon_j)]$ with Type-I EV shocks. We iterate until the sup-norm $\|V^{(k+1)} - V^{(k)}\|_\infty < \text{tol}$.

**State space**: $(\ell, d, e) \in \{1,2\} \times \{0,\ldots,D_{\max}\} \times \{0,\ldots,3\}$ — only $2 \times 3 \times 4 = 24$ states for $J=2,\, D_{\max}=2$, so VFI is very fast.

In [43]:
def solve_vfi(alpha, gamma, beta_sc, beta_dep,
              tol: float = 1e-10, max_iter: int = 2_000) -> np.ndarray:
    """
    Solve the consumer's infinite-horizon stationary Bellman equation by VFI.

    This is the *inner loop* of NFXP: given structural parameters, find the
    value function fixed point V* such that T(V*) = V*.

    With i.i.d. Type-I EV shocks the Bellman operator is:
        T(V)[l,d,e] = log Σ_j exp( u(j,l,d,e) + δ·EV(j,l,d,e) )

    where EV(j,l,d,e) = Σ_{e'} P(e'|e) · V( l'(j,l,d), d'(j,l,d), e' )
    and (l'(j,l,d), d'(j,l,d)) is the deterministic next state (eq. 2.1).

    Returns
    -------
    V : ndarray, shape (J, D_MAX+1, N_PROMO) — ex-ante value function
    """
    # Initialise value function at zero
    V = np.zeros((J, D_MAX + 1, N_PROMO))

    for _ in range(max_iter):
        # ── Expected continuation value ──────────────────────────────────────
        # Integrate V over next period's promotion state using the Markov matrix.
        # V is (J, D_MAX+1, N_PROMO) → reshape to (J*(D_MAX+1), N_PROMO)
        EV = (V.reshape(J * (D_MAX + 1), N_PROMO) @ PROMO_TRANS.T
              ).reshape(J, D_MAX + 1, N_PROMO)

        V_new = np.empty_like(V)

        for l_idx in range(J):
            ell = l_idx + 1
            for d in range(D_MAX + 1):
                d_next = min(d + 1, D_MAX)   # duration if no purchase
                for e in range(N_PROMO):
                    # Choice-specific values Q(j) = u(j) + δ·EV(next state)
                    Q = np.empty(J + 1)

                    # j=0: no purchase → state remains (ell, d_next)
                    Q[0] = (flow_util(0, ell, d, e, alpha, gamma, beta_sc, beta_dep)
                            + DELTA * EV[l_idx, d_next, e])

                    # j>0: buy brand j → state resets to (j, 0)
                    for j_idx in range(J):
                        Q[j_idx + 1] = (
                            flow_util(j_idx + 1, ell, d, e, alpha, gamma, beta_sc, beta_dep)
                            + DELTA * EV[j_idx, 0, e]
                        )

                    # Log-sum-exp = E[max(Q + ε)] for EV1 shocks (closed form)
                    q_max = Q.max()
                    V_new[l_idx, d, e] = q_max + np.log(np.exp(Q - q_max).sum())

        # Convergence check (sup-norm on value function)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new

        V = V_new

    # Return best iterate if max_iter is reached without convergence
    return V

## 5. Conditional choice probabilities (CCPs)

Given the solved value function $V^*$ we compute CCPs as the multinomial logit softmax over the choice-specific values $Q(j) = u(j) + \delta \cdot EV(\text{next state})$.

In [44]:
def compute_ccps(V, alpha, gamma, beta_sc, beta_dep) -> np.ndarray:
    """
    Compute the (J, D_MAX+1, N_PROMO, J+1) array of CCPs from the solved
    value function.

    P[l_idx, d, e, j] = Pr(choose j | state (ell, d, e))
    """
    # Expected continuation value (same computation as in VFI)
    EV = (V.reshape(J * (D_MAX + 1), N_PROMO) @ PROMO_TRANS.T
          ).reshape(J, D_MAX + 1, N_PROMO)

    P = np.empty((J, D_MAX + 1, N_PROMO, J + 1))

    for l_idx in range(J):
        ell = l_idx + 1
        for d in range(D_MAX + 1):
            d_next = min(d + 1, D_MAX)
            for e in range(N_PROMO):
                Q = np.empty(J + 1)
                Q[0] = (flow_util(0, ell, d, e, alpha, gamma, beta_sc, beta_dep)
                        + DELTA * EV[l_idx, d_next, e])
                for j_idx in range(J):
                    Q[j_idx + 1] = (
                        flow_util(j_idx + 1, ell, d, e, alpha, gamma, beta_sc, beta_dep)
                        + DELTA * EV[j_idx, 0, e]
                    )
                # Softmax (numerically stable via max subtraction)
                q_shifted = Q - Q.max()
                w = np.exp(q_shifted)
                P[l_idx, d, e, :] = w / w.sum()

    return P

## 6. Data simulation

We simulate a consumer panel from the true CCPs. The state transition follows equation (2.1): if the consumer buys brand $j$, duration resets to 0 and $\ell$ is set to $j$; otherwise duration increments (capped at $D_{\max}$).

In [45]:
def _sample_rows(rng, row_probs: np.ndarray) -> np.ndarray:
    """
    Vectorised categorical draw: for each row i in row_probs (shape N×K),
    draw one index from the corresponding probability vector.

    Uses the inverse-CDF trick:
        k* = number of cumsum entries strictly less than u_i
    which gives the unique k with CDF[k-1] < u_i <= CDF[k].
    """
    u = rng.random(len(row_probs))           # (N,) uniform draws
    cumsum = np.cumsum(row_probs, axis=1)    # (N, K) running totals
    return (u[:, None] > cumsum).sum(axis=1)


def simulate_panel(P_true: np.ndarray,
                   n_consumers: int = N,
                   n_periods: int = T,
                   seed=None) -> dict:
    """
    Simulate a consumer panel from the model given true CCPs P_true.

    Returns a dict with four (n_consumers × n_periods) integer arrays:
        Y     — observed choice in {0,...,J}
        L     — state ell (last brand purchased) at the start of period t
        D     — state d (duration) at the start of period t
        E_IDX — promotion state index at period t
    """
    rng = np.random.default_rng(seed)

    Y     = np.zeros((n_consumers, n_periods), dtype=int)
    L     = np.zeros((n_consumers, n_periods), dtype=int)
    D     = np.zeros((n_consumers, n_periods), dtype=int)
    E_IDX = np.zeros((n_consumers, n_periods), dtype=int)

    # Initialise states randomly
    ell   = rng.integers(1, J + 1, size=n_consumers)
    dur   = rng.integers(0, D_MAX + 1, size=n_consumers)
    e_idx = rng.integers(0, N_PROMO, size=n_consumers)

    for t in range(n_periods):
        # Record beginning-of-period states
        L[:, t]     = ell
        D[:, t]     = dur
        E_IDX[:, t] = e_idx

        # Look up each consumer's CCP row and draw a choice
        probs = P_true[ell - 1, np.minimum(dur, D_MAX), e_idx, :]   # (N, J+1)
        y = _sample_rows(rng, probs)
        Y[:, t] = y

        # Update endogenous states (transition rule, equation 2.1)
        bought = y > 0
        ell = np.where(bought, y, ell)                               # new last brand
        dur = np.where(bought, 0, np.minimum(dur + 1, D_MAX))        # reset or increment

        # Update exogenous promotion state via the Markov chain
        e_idx = _sample_rows(rng, PROMO_TRANS[e_idx])

    return {"Y": Y, "L": L, "D": D, "E_IDX": E_IDX}

## 7. Log-likelihood and NFXP objective

The log-likelihood sums over all $(i,t)$ observations: $\mathcal{L}(\theta) = \sum_{i,t} \log P(y_{it} \mid \ell_{it}, d_{it}, e_{it};\, \theta)$.

The NFXP objective **minimises the negative log-likelihood** by calling VFI at each candidate $\theta$.

In [46]:
def log_likelihood(data: dict, P: np.ndarray) -> float:
    """
    Vectorised log-likelihood summed over all (consumer, period) pairs.

    Uses NumPy advanced indexing to avoid Python loops:
        probs[i,t] = P[ L[i,t]-1, D[i,t], E[i,t], Y[i,t] ]
    """
    Y, L, D, E = data["Y"], data["L"], data["D"], data["E_IDX"]
    probs = P[L - 1, D, E, Y]                          # (N, T)
    return float(np.sum(np.log(np.maximum(probs, 1e-300))))


def unpack(theta: np.ndarray):
    """
    Unpack the theta vector into named parameter arrays.

    theta = [alpha_2, gamma, beta_sc_12, beta_sc_21, beta_dep_1, beta_dep_2]
    alpha_1 = 0 is the normalisation (brand 1 is the reference brand).
    """
    alpha    = np.array([0.0, theta[0]])        # alpha_1=0, alpha_2=theta[0]
    gamma    = float(theta[1])
    beta_sc  = np.array([[0.0,    theta[2]],    # from brand 1: to brand 2
                          [theta[3], 0.0   ]])   # from brand 2: to brand 1
    beta_dep = np.array([theta[4], theta[5]])
    return alpha, gamma, beta_sc, beta_dep


def nfxp_objective(theta: np.ndarray, data: dict) -> float:
    """
    NFXP objective: negative log-likelihood as a function of theta.

    Algorithm
    ---------
    1. Unpack theta into structural parameters.
    2. *Inner loop*: run VFI to convergence → V*(theta).
    3. Compute CCPs from V*(theta).
    4. Evaluate the log-likelihood on the observed data.
    """
    alpha, gamma, beta_sc, beta_dep = unpack(theta)
    V = solve_vfi(alpha, gamma, beta_sc, beta_dep)      # ← inner loop
    P = compute_ccps(V, alpha, gamma, beta_sc, beta_dep)
    return -log_likelihood(data, P)


def estimate_nfxp(data: dict, theta0: np.ndarray = None, verbose: bool = False):
    """
    Estimate structural parameters by NFXP using Nelder-Mead.

    Nelder-Mead (gradient-free simplex method) is used because:
      - VFI introduces small numerical errors that can mislead
        finite-difference gradient approximations.
      - The parameter space is small (6 parameters), so the method is tractable.

    Parameters
    ----------
    data   : panel dict from simulate_panel
    theta0 : (6,) starting values; defaults to small positive vector
    verbose: bool; display optimizer output

    Returns scipy OptimizeResult
    """
    if theta0 is None:
        theta0 = np.array([0.1, 0.3, 0.3, 0.3, 0.1, 0.1])

    return minimize(
        fun=nfxp_objective,
        x0=theta0,
        args=(data,),
        method="Nelder-Mead",
        options={
            "maxiter":  10_000,
            "xatol":    1e-5,       # tolerance on parameter change
            "fatol":    1e-5,       # tolerance on function value change
            "disp":     verbose,
            "adaptive": True,       # adaptive simplex scales better with dimensionality
        },
    )

## 8. Pilot — single panel

We simulate one panel, estimate, and inspect the results. The pilot is useful for verifying that VFI converges and the estimator recovers the true parameters before running the full MC.

In [47]:
# ── Step 1: solve the DP at the true parameters ───────────────────────────────
print("Solving DP at true parameters (VFI)...")
alpha0, g0, sc0, dep0 = unpack(THETA_TRUE)
V_true = solve_vfi(alpha0, g0, sc0, dep0)
P_true = compute_ccps(V_true, alpha0, g0, sc0, dep0)
print("  VFI converged.")

# ── Step 2: simulate one panel ────────────────────────────────────────────────
print("Simulating panel (N={}, T={})...".format(N, T))
data_pilot = simulate_panel(P_true, seed=42)
Y = data_pilot["Y"]

purchase_rate = (Y > 0).mean()
brand_shares  = [(Y == j).mean() for j in range(J + 1)]
print(f"  Purchase rate : {purchase_rate:.1%}")
print(f"  Brand shares  : no purchase={brand_shares[0]:.2f}  "
      + "  ".join(f"brand {j}={brand_shares[j]:.2f}" for j in range(1, J + 1)))

# ── Step 3: estimate by NFXP ─────────────────────────────────────────────────
print("\nEstimating by NFXP (Nelder-Mead)...")
t0 = time.perf_counter()
# Starting values: true theta + small perturbation (tests robustness)
theta0_pilot = THETA_TRUE + np.array([0.05, 0.05, 0.05, 0.05, 0.02, 0.02])
result_pilot = estimate_nfxp(data_pilot, theta0=theta0_pilot, verbose=True)
t_pilot = time.perf_counter() - t0

print(f"\n  Time      : {t_pilot:.1f}s")
print(f"  Converged : {result_pilot.success}  (iterations={result_pilot.nit})")

# ── Results table ─────────────────────────────────────────────────────────────
df_pilot = pd.DataFrame({
    "Parameter": PARAM_NAMES,
    "True":      THETA_TRUE,
    "NFXP":      result_pilot.x,
    "Bias":      result_pilot.x - THETA_TRUE,
})
print("\n" + df_pilot.to_string(index=False, float_format=lambda x: f"{x:+.4f}" if x != 0 else f"{x:.4f}"))

Solving DP at true parameters (VFI)...
  VFI converged.
Simulating panel (N=2000, T=52)...
  Purchase rate : 97.2%
  Brand shares  : no purchase=0.03  brand 1=0.02  brand 2=0.95

Estimating by NFXP (Nelder-Mead)...
Optimization terminated successfully.
         Current function value: 12330.951411
         Iterations: 264
         Function evaluations: 459

  Time      : 502.4s
  Converged : True  (iterations=264)

 Parameter    True    NFXP    Bias
   alpha_2 +0.3000 +0.3121 +0.0121
     gamma -0.5000 -0.4977 +0.0023
beta_sc_12 +0.5500 +0.4769 -0.0731
beta_sc_21 +0.5000 +0.5292 +0.0292
beta_dep_1 +0.2500 +0.3797 +0.1297
beta_dep_2 +0.2500 +0.2386 -0.0114


## 9. Monte Carlo simulation

Vi kører `MC_REPS` uafhængige replikationer. For hver kørsel:
1. Simulerer et nyt panel fra de sande CCP'er (forskellig seed)
2. Estimerer $\theta$ med NFXP (startværdier = sand $\theta$ + lille støj)
3. Gemmer estimat, bias og kvadratfejl

**Tip:** Reducer `MC_REPS` eller `N` øverst i notebook'en hvis kørselstiden er for lang (~1 min pr. replikation).

In [48]:
def run_monte_carlo(n_reps: int = MC_REPS, seed: int = MC_SEED):
    """
    Kører hele Monte Carlo-eksperimentet.

    For hver replikation:
      1. Simulér panel fra de sande DGP-CCP'er.
      2. Estimér theta med NFXP.
      3. Registrér estimater, bias og kvadratfejl.

    Returnerer
    ----------
    results_df : DataFrame med én række pr. (replikation × parameter)
    summary_df : DataFrame med mean bias, std.afv. og RMSE pr. parameter
    """
    rng_master = np.random.default_rng(seed)
    rep_seeds  = rng_master.integers(0, 1_000_000, size=n_reps)

    # Løs DP én gang ved de sande parametre (alle rep. bruger samme P_true)
    alpha_t, g_t, sc_t, dep_t = unpack(THETA_TRUE)
    V_true = solve_vfi(alpha_t, g_t, sc_t, dep_t)
    P_true = compute_ccps(V_true, alpha_t, g_t, sc_t, dep_t)

    print(f"Monte Carlo  |  J={J}, T={T}, N={N}, D_MAX={D_MAX}, reps={n_reps}\n")

    rows = []
    for rep in range(1, n_reps + 1):
        # Simulér panel for denne replikation
        data = simulate_panel(P_true, n_consumers=N, n_periods=T,
                              seed=int(rep_seeds[rep - 1]))

        # Startværdier: perturber sand theta let (replikations-specifik seed)
        rng_s = np.random.default_rng(int(rep_seeds[rep - 1]) + 999)
        theta0 = THETA_TRUE + rng_s.normal(0.0, 0.05, size=len(THETA_TRUE))

        # Estimér
        t0     = time.perf_counter()
        result = estimate_nfxp(data, theta0=theta0)
        t_sec  = time.perf_counter() - t0

        print(f"  Rep {rep:>3}/{n_reps}  konvergeret={result.success}"
              f"  nit={result.nit:>5}  tid={t_sec:>5.1f}s")

        # Gem resultater pr. parameter
        for k, name in enumerate(PARAM_NAMES):
            rows.append({
                "replikation": rep,
                "parameter":   name,
                "sand":        THETA_TRUE[k],
                "estimat":     result.x[k],
                "bias":        result.x[k] - THETA_TRUE[k],
                "kv_fejl":     (result.x[k] - THETA_TRUE[k]) ** 2,
                "konvergeret": int(result.success),
                "n_iter":      result.nit,
                "tid_sek":     t_sec,
            })

    results_df = pd.DataFrame(rows)

    # Opsummerende statistik pr. parameter
    summary_rows = []
    for name in PARAM_NAMES:
        sub = results_df[results_df["parameter"] == name]
        est = sub["estimat"].to_numpy()
        summary_rows.append({
            "parameter":  name,
            "sand":       THETA_TRUE[PARAM_NAMES.index(name)],
            "mean_est":   est.mean(),
            "bias":       sub["bias"].mean(),
            "std_afv":    est.std(ddof=1),
            "rmse":       np.sqrt(sub["kv_fejl"].mean()),
            "konv_rate":  sub["konvergeret"].mean(),
        })
    summary_df = pd.DataFrame(summary_rows)

    return results_df, summary_df


# Kør Monte Carlo (kan tage ~20 min ved MC_REPS=20, N=2000)
results_df, summary_df = run_monte_carlo()

Monte Carlo  |  J=2, T=52, N=2000, D_MAX=20, reps=20

  Rep   1/20  konvergeret=True  nit=  207  tid=418.2s
  Rep   2/20  konvergeret=True  nit=  280  tid=513.1s
  Rep   3/20  konvergeret=True  nit=  229  tid=433.7s
  Rep   4/20  konvergeret=True  nit=  228  tid=425.9s
  Rep   5/20  konvergeret=True  nit=  270  tid=512.7s
  Rep   6/20  konvergeret=True  nit=  299  tid=549.4s
  Rep   7/20  konvergeret=True  nit=  235  tid=437.7s
  Rep   8/20  konvergeret=True  nit=  265  tid=489.0s
  Rep   9/20  konvergeret=True  nit=  267  tid=496.1s
  Rep  10/20  konvergeret=True  nit=  215  tid=414.6s
  Rep  11/20  konvergeret=True  nit=  269  tid=500.4s
  Rep  12/20  konvergeret=True  nit=  236  tid=454.8s
  Rep  13/20  konvergeret=True  nit=  212  tid=416.8s
  Rep  14/20  konvergeret=True  nit=  256  tid=496.5s
  Rep  15/20  konvergeret=True  nit=  217  tid=418.6s
  Rep  16/20  konvergeret=True  nit=  229  tid=443.3s
  Rep  17/20  konvergeret=True  nit=  254  tid=492.8s
  Rep  18/20  konvergeret=Tr

## 10. Resultater og visualisering

In [49]:
# ── MC-opsummeringsstabel ──────────────────────────────────────────────────────
print("Monte Carlo opsummering")
print("=" * 65)
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("=" * 65)

# Gem til CSV
results_df.to_csv("nfxp_mc_results.csv", index=False)
summary_df.to_csv("nfxp_mc_summary.csv", index=False)
print("\nGemt: nfxp_mc_results.csv  og  nfxp_mc_summary.csv")

Monte Carlo opsummering
 parameter    sand  mean_est    bias  std_afv   rmse  konv_rate
   alpha_2  0.3000    0.3082  0.0082   0.0333 0.0334     1.0000
     gamma -0.5000   -0.5006 -0.0006   0.0160 0.0156     1.0000
beta_sc_12  0.5500    0.5641  0.0141   0.1205 0.1183     1.0000
beta_sc_21  0.5000    0.4928 -0.0072   0.1096 0.1071     1.0000
beta_dep_1  0.2500    0.2527  0.0027   0.0390 0.0381     1.0000
beta_dep_2  0.2500    0.2513  0.0013   0.0289 0.0282     1.0000

Gemt: nfxp_mc_results.csv  og  nfxp_mc_summary.csv
